# Phần 10 — Khóa cấu hình và chung kết 3 seed
**2A202602810 · Nguyễn Thái Anh**

Final **F01 = recipe T04 CutMix + I07 temperature scaling**, baseline **T00 = recipe B03 + I00**.
Cùng ConvNeXt-Tiny, pretrained tag, fold 0, 12 epoch, batch 64, input 224; seed **0,1,2**.
Seed 0 dùng lại T04/B03 đã hoàn tất; chỉ train thêm **4 run** độc lập ở seed 1 và 2.

**Run All với các flag mặc định sẽ train 4 run còn thiếu, khóa cả 6 checkpoint/nhiệt độ,
rồi đánh giá toàn bộ test đúng một lượt cho mỗi model và chạy evaluator gốc.**
Chọn kernel `.venv` trên host GPU. Không đổi recipe/seed/preprocessing sau khi đã mở test.
Muốn chỉ train/fit validation trước: đặt `RUN_TEST=False`, `RUN_EVALUATOR=False`.
Muốn chỉ xem: đặt cả 3 flag False. Khuyến nghị chạy tuần tự trên một GPU.

Những bản final calibrated/uncalibrated lấy từ cùng logits test, không forward thêm.
Trường hợp test đã bắt đầu nhưng chưa lưu đủ logits được chặn thử lại; giữ marker/log để xử lý theo protocol lớp.

## 1. Môi trường

In [ ]:
import os, sys, json
from pathlib import Path
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None: raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("MPLCONFIGDIR",str(ROOT / ".cache/matplotlib"))
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(CODE))
import pandas as pd
from IPython.display import display, Image, Markdown
import final_notebook as fn
import train as trainer
assert Path(trainer.__file__).resolve() == CODE / "train.py"
FINAL_ROOT = fn.root_path()
print("ROOT:", ROOT)
print("Python:", sys.executable)
print("Final output:", FINAL_ROOT)

## 2. Các giai đoạn chạy
Cấu hình numeric trong manifest đã khóa; các flag chỉ chọn giai đoạn thực hiện.
Training không tạo test prediction; test chỉ được gọi khi đủ 6 training/validation preparation.
Ngắt khi train: resume cùng config. Ngắt sau khi lưu đủ logits test: xuất lại từ cache, không chạy model lần nữa.

In [ ]:
RUN_TRAINING = True
RUN_TEST = True
RUN_EVALUATOR = True
print("Training:", RUN_TRAINING, "Test:", RUN_TEST, "Evaluator:", RUN_EVALUATOR)

## 3. Manifest khóa trước test
Khóa recipe/optimizer/scheduler/checkpoint rule, tag, preprocessing, phương pháp suy luận,
protocol fit T từng seed trên val, seed list, CSV hashes, evidence phần 9 và code/evaluator hashes.
Manifest checksum và nguồn checkpoint seed 0 được kiểm tra khi chạy lại. Code/config đã khóa không sửa giữa chừng.

In [ ]:
manifest = None
if (FINAL_ROOT / "manifest.json").exists():
    manifest = fn.load_manifest(FINAL_ROOT)
elif RUN_TRAINING or RUN_TEST or RUN_EVALUATOR:
    manifest = fn.lock_configuration(FINAL_ROOT)
if manifest is not None:
    display({"locked_at_utc":manifest["locked_at_utc"],"final":manifest["spec"]["final"],
             "baseline":manifest["spec"]["baseline"],"seeds":manifest["spec"]["seeds"],
             "checkpoint_rule":manifest["spec"]["checkpoint_selection"],
             "inference":manifest["spec"]["inference"]})
    display(pd.DataFrame([{k:v for k,v in e.items() if k != "source_cfg"}
                          for e in manifest["spec"]["entries"]]))
else:
    print("Chưa khóa cấu hình; bật các flag để tạo manifest trước khi chạy.")

## 4. Tiến độ trước chạy
T00/seed0 → B03/seed0; F01/seed0 → T04/seed0, không giả lập hai lần train mới.
Source predictions thô của seed 1/2 nằm trong predictions/training/;
F01_seed*_val.csv ở predictions/ là bản đã hiệu chuẩn, không ghi đè raw CSV của trainer.

In [ ]:
progress = fn.collect_progress(FINAL_ROOT)
display(progress)
if (FINAL_ROOT / "completion.json").exists():
    display(json.loads((FINAL_ROOT / "completion.json").read_text()))
else:
    print("Phần 10 chưa hoàn tất; xem trạng thái 6 mục trong bảng.")

## 5. Chạy toàn bộ giai đoạn đã chọn
Log stream từ process con. Training mỗi seed reset RNG/model/optimizer; không truyền trọng số seed trước.
Sau khi chuẩn bị đủ 6 mục, checkpoint_set.json khóa tất cả checkpoint + temperature trước test.
Test đọc đủ 3.507 ảnh test gốc, FP32 single view; F01 chia logits bằng T đã fit val, T00 giữ T=1.
Test marker tạo ngay trước lượt forward, raw logits lưu một lần; mọi lần chạy lại chỉ audit/xuất cache.
Nếu code/source/config đổi hoặc marker có nhưng raw cache chưa đủ, runner dừng và không tự mở test lại.

In [ ]:
if RUN_TRAINING or RUN_TEST or RUN_EVALUATOR:
    progress = fn.run_in_subprocess(root=str(FINAL_ROOT),run_training=RUN_TRAINING,
                                    run_test=RUN_TEST,run_evaluator=RUN_EVALUATOR)
else:
    progress = fn.collect_progress(FINAL_ROOT)
display(progress)

## 6. Kiểm tra validation và nhiệt độ từng seed
Scalar T fit riêng mỗi final seed, cùng protocol đã khóa, accuracy/macro-F1 giữ nguyên.
Baseline T00 không hiệu chuẩn. NLL/ECE val không bảo đảm cải thiện tương tự trên test.

In [ ]:
prepared_rows = []
if manifest is not None:
    for e in manifest["spec"]["entries"]:
        path = fn.entry_root(FINAL_ROOT,e["role"],e["seed"]) / "prepared.json"
        if path.exists():
            p = fn.verify_prepared(FINAL_ROOT,manifest,e["role"],e["seed"])
            prepared_rows.append({"role":e["role"],"seed":e["seed"],"source_id":p["source_id"],
                "T":p["temperature"],"val_macro_f1":p["val"]["macro_f1"],
                "val_ece_before":p["val_uncalibrated"]["ece"],"val_ece_after":p["val"]["ece"]})
display(pd.DataFrame(prepared_rows))

## 7. Learning curves từ các run thật
Curve seed 0 giữ ID B03/T04; seed 1/2 có ID T00/F01. Không dùng curve theo epoch làm std theo seed.

In [ ]:
if manifest is not None:
    for e in manifest["spec"]["entries"]:
        cfg = trainer.Config(**e["source_cfg"])
        curve = Path(cfg.curves_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{cfg.backbone}.png"
        if curve.exists():
            print(e["role"], "seed",e["seed"],"source",cfg.exp_id)
            display(Image(filename=str(curve)))

## 8. Evaluator gốc và mean ± std mẫu
F01/T00 đủ 3 seed; calibrated và uncalibrated dùng cùng logits/model/filename order.
score/grade được gọi qua eval.py gốc, std ddof=1; không sửa công thức hoặc nhập metric bằng tay.
Latency grade lấy benchmark I07 seed 0 của phần 9, đại diện cùng kiến trúc/pipeline locked,
không phải latency ensemble 3 model: 3 seed là 3 lần đánh giá độc lập.

In [ ]:
stats = []
eval_out = FINAL_ROOT / "eval_out"
for role in ("F01","T00"):
    path = eval_out / f"{role}_summary.json"
    if path.exists():
        values = json.loads(path.read_text())
        for metric in ("macro_f1","top1","balanced_acc","ece","nll"):
            stats.append({"role":role,"metric":metric,"n_seeds":len(values["seeds"]),
                          "mean":values[metric]["mean"],"std_ddof1":values[metric]["std"]})
display(pd.DataFrame(stats))
if (eval_out / "grade_I.json").exists():
    grade = json.loads((eval_out / "grade_I.json").read_text())
    display(pd.DataFrame(grade["items"]))
    print("Điểm gợi ý rubric I:", grade["total"], "/", grade["max_scored"])
else:
    print("Chưa đủ kết quả chung kết để score/grade.")

## 9. Per-class và confusion của final
Confusion sum cộng qua 3 seed, support tổng 3×3.507; không trình bày như một seed.
Per-class mean/std lấy evaluator từng seed; không lấy F1 từ confusion sum làm mean F1.

In [ ]:
for name in ("F01_per_seed.csv","T00_per_seed.csv","F01_per_class.csv","F01_confusion_sum.csv"):
    path = eval_out / name
    if path.exists():
        print(name)
        display(pd.read_csv(path))

## 10. Kết luận phần 10
Chỉ hoàn tất khi đủ 6 test predictions đã audit và score/grade thành công, có completion.json.
Sau test giữ nguyên quyết định; không chọn recipe/TTA/T dựa trên test.
Tiếp theo phần 11–13: Excel 7 sheet, báo cáo, confusion/error/reliability/tradeoff,
README tái lập và đối chiếu sản phẩm nộp. Chi tiết thao tác nằm trong README bài làm (submissions/2A202602810_NguyenThaiAnh/README.md).

In [ ]:
progress = fn.collect_progress(FINAL_ROOT)
display(progress)
if (FINAL_ROOT / "completion.json").exists():
    display(json.loads((FINAL_ROOT / "completion.json").read_text()))
else:
    print("Phần 10 chưa hoàn tất; xem trạng thái 6 mục trong bảng.")